
**Part 1 of 2: Getting and Cleaning the Data**

**Research Question**:
What are the most commonly reported adverse events for Ozempic compared to Metformin,
and how do the severity and patient demographics differ between these two drugs?

**Why this matters to me**:
Ozempic has become extremely popular recently, not just for diabetes but also for weight loss.
Meanwhile, Metformin has been the go-to diabetes medication for decades. I want to understand
what side effects people are actually experiencing with these drugs and whether the newer drug
(Ozempic) has a different safety profile than the traditional option (Metformin).


In [1]:
import requests
import pandas as pd
import json
import time
from datetime import datetime

print("Libraries imported successfully!")


Libraries imported successfully!


#  1. Setting Up Functions to Talk to the OpenFDA API

The OpenFDA API gives us data in JSON format, which is basically nested dictionaries.
I need to write functions that can request this data and handle the fact that the API
only gives us 1000 records at a time (so I'll need to make multiple requests).



    This function sends a request to the OpenFDA API and gets back adverse event reports.
    
    How it works:
    - search_term: tells the API which drug I'm looking for
    - limit: how many records to get (max is 1000 per request)
    - skip: lets me get the next batch of records (for pagination)

In [2]:
def fetch_openfda_data(search_term, limit=1000, skip=0):
    """
    This function sends a request to the OpenFDA API and gets back adverse event reports.

    How it works:
    - search_term: tells the API which drug I'm looking for
    - limit: how many records to get (max is 1000 per request)
    - skip: lets me get the next batch of records (for pagination)
    """
    base_url = "https://api.fda.gov/drug/event.json"

    # Construct the search query
    params = {
        'search': search_term,
        'limit': limit,
        'skip': skip
    }

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()  # Raise an error for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return None

def collect_multiple_pages(search_term, total_records=5000):
    """
    Since the API only gives 1000 records at a time, I need to call it multiple times.
    This function handles that by making several requests and combining all the results.
    I'm also adding a 1-second delay between requests to avoid overwhelming the API.
    """
    all_results = []
    limit = 1000  # API max per request

    for skip in range(0, total_records, limit):
        print(f"Fetching records {skip} to {skip + limit}...")
        data = fetch_openfda_data(search_term, limit=limit, skip=skip)

        if data and 'results' in data:
            all_results.extend(data['results'])
            print(f"  Retrieved {len(data['results'])} records")
        else:
            print(f"  No more data available or error occurred")
            break

        # Be polite to the API - add a small delay between requests
        time.sleep(1)

    print(f"Total records collected: {len(all_results)}")
    return all_results

# 2. Collecting Ozempic Data

 Now I'll use my functions to actually get the data. I'm searching for reports where
 Ozempic was listed as a drug. The API uses brand names, so I search for "Ozempic" specifically.
 I'm collecting 5,000 reports to have enough data to analyze.

In [3]:
print("COLLECTING OZEMPIC DATA")
print("="*70)

# Search for Ozempic (brand name)
ozempic_search = 'patient.drug.openfda.brand_name:"Ozempic"'
ozempic_data = collect_multiple_pages(ozempic_search, total_records=5000)

print(f"\nOzempic data collection complete: {len(ozempic_data)} reports")


COLLECTING OZEMPIC DATA
Fetching records 0 to 1000...
  Retrieved 1000 records
Fetching records 1000 to 2000...
  Retrieved 1000 records
Fetching records 2000 to 3000...
  Retrieved 1000 records
Fetching records 3000 to 4000...
  Retrieved 1000 records
Fetching records 4000 to 5000...
  Retrieved 1000 records
Total records collected: 5000

Ozempic data collection complete: 5000 reports


 # 3. Collecting Metformin Data

Same process as Ozempic, but now I'm searching for Metformin. I'm using the generic name
instead of a brand name because Metformin is sold under many different brands (Glucophage, etc.)
 and I want to capture all of them.

In [4]:
print("\n" + "="*70)
print("COLLECTING METFORMIN DATA")
print("="*70)

# Search for Metformin (generic name to capture all brands)
metformin_search = 'patient.drug.openfda.generic_name:"metformin"'
metformin_data = collect_multiple_pages(metformin_search, total_records=5000)

print(f"\nMetformin data collection complete: {len(metformin_data)} reports")


COLLECTING METFORMIN DATA
Fetching records 0 to 1000...
  Retrieved 1000 records
Fetching records 1000 to 2000...
  Retrieved 1000 records
Fetching records 2000 to 3000...
  Retrieved 1000 records
Fetching records 3000 to 4000...
  Retrieved 1000 records
Fetching records 4000 to 5000...
  Retrieved 1000 records
Total records collected: 5000

Metformin data collection complete: 5000 reports


 # 4. Parsing the JSON Data

The data from the API is in JSON format with lots of nested fields. It's messy and hard to analyze.
I need to extract the specific fields I care about and put them into a flat table (DataFrame).

## What I'm extracting:
- Basic info: report ID, date received, whether it was serious
 - Serious outcomes: death, hospitalization, disability, life-threatening
- Patient info: age, sex, weight
- The actual side effects (reactions) reported
 - Who reported it (doctor, patient, pharmacist, etc.)

In [5]:
print("\n" + "="*70)
print("PARSING JSON DATA INTO STRUCTURED FORMAT")
print("="*70)

def parse_adverse_event(record, drug_name):
    """
    This function takes one messy JSON record and pulls out the fields I need.
    It creates a simple dictionary with clear field names that will become one row in my DataFrame.
    """
    parsed = {}

    # Add drug identifier
    parsed['drug'] = drug_name

    # Report metadata
    parsed['report_id'] = record.get('safetyreportid', None)
    parsed['receive_date'] = record.get('receivedate', None)
    parsed['serious'] = record.get('serious', None)

    # Serious outcomes (if serious = 1)
    parsed['serious_death'] = record.get('seriousnessdeath', 0)
    parsed['serious_hospitalization'] = record.get('seriousnesshospitalization', 0)
    parsed['serious_disabling'] = record.get('seriousnessdisabling', 0)
    parsed['serious_life_threatening'] = record.get('seriousnesslifethreatening', 0)

    # Patient information
    patient = record.get('patient', {})
    parsed['patient_age'] = patient.get('patientonsetage', None)
    parsed['patient_age_unit'] = patient.get('patientonsetageunit', None)
    parsed['patient_sex'] = patient.get('patientsex', None)
    parsed['patient_weight'] = patient.get('patientweight', None)

    # Reactions (adverse events) - get first 3 reactions
    reactions = patient.get('reaction', [])
    reaction_terms = [r.get('reactionmeddrapt', '') for r in reactions[:3]]
    parsed['reaction_1'] = reaction_terms[0] if len(reaction_terms) > 0 else None
    parsed['reaction_2'] = reaction_terms[1] if len(reaction_terms) > 1 else None
    parsed['reaction_3'] = reaction_terms[2] if len(reaction_terms) > 2 else None

    # Primary source (reporter type)
    primary_source = record.get('primarysource', {})
    parsed['reporter_qualification'] = primary_source.get('qualification', None)

    # Country
    parsed['country'] = record.get('occurcountry', None)

    return parsed

# Now I'll apply this parsing function to all my Ozempic and Metformin data
print("Parsing Ozempic records...")
ozempic_parsed = [parse_adverse_event(record, 'Ozempic') for record in ozempic_data]
ozempic_df = pd.DataFrame(ozempic_parsed)
print(f"Ozempic DataFrame created: {ozempic_df.shape[0]} rows, {ozempic_df.shape[1]} columns")

print("Parsing Metformin records...")
metformin_parsed = [parse_adverse_event(record, 'Metformin') for record in metformin_data]
metformin_df = pd.DataFrame(metformin_parsed)
print(f"Metformin DataFrame created: {metformin_df.shape[0]} rows, {metformin_df.shape[1]} columns")


PARSING JSON DATA INTO STRUCTURED FORMAT
Parsing Ozempic records...
Ozempic DataFrame created: 5000 rows, 17 columns
Parsing Metformin records...
Metformin DataFrame created: 5000 rows, 17 columns


# 5. Combining and Cleaning the Data

Now that I have two clean DataFrames, I'll combine them into one dataset.
But there's still work to do - dates are stored as strings, ages are in different units
 (years, months, days, etc.), and codes need to be converted to readable labels.

In [6]:
print("\n" + "="*70)
print("COMBINING AND CLEANING DATA")
print("="*70)

# Combine both dataframes
df = pd.concat([ozempic_df, metformin_df], ignore_index=True)
print(f"\nCombined dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

print("\nFirst 5 rows of combined data:")
print(df.head())


COMBINING AND CLEANING DATA

Combined dataset shape: 10000 rows, 17 columns

First 5 rows of combined data:
      drug report_id receive_date serious serious_death  \
0  Ozempic  10188271     20140522       1             0   
1  Ozempic  10398782     20140821       2             2   
2  Ozempic  10515213     20141014       1             0   
3  Ozempic  10787292     20150211       1             0   
4  Ozempic  11104986     20150511       1             2   

  serious_hospitalization serious_disabling serious_life_threatening  \
0                       1                 1                        0   
1                       2                 2                        2   
2                       0                 0                        0   
3                       0                 0                        0   
4                       2                 2                        2   

  patient_age patient_age_unit patient_sex patient_weight  \
0          81              801           2

In [12]:

print("\n" + "-"*70)
print("CLEANING STEPS:")
print("-"*70)

# **Step 1: Fix the dates**
# The dates come as strings like "20230415" and I need proper datetime objects
print("\n1. Converting receive_date to datetime format...")
df['receive_date'] = pd.to_datetime(df['receive_date'], format='%Y%m%d', errors='coerce')
print(f"   Conversion complete. Sample dates: {df['receive_date'].dropna().head().tolist()}")

# **Step 2: Convert string columns to numeric**
# The API returns everything as strings, so I need to convert to numbers first
print("\n2. Converting string columns to numeric...")
df['patient_age'] = pd.to_numeric(df['patient_age'], errors='coerce')
df['patient_age_unit'] = pd.to_numeric(df['patient_age_unit'], errors='coerce')
df['patient_sex'] = pd.to_numeric(df['patient_sex'], errors='coerce')
df['reporter_qualification'] = pd.to_numeric(df['reporter_qualification'], errors='coerce')
df['serious_death'] = pd.to_numeric(df['serious_death'], errors='coerce').fillna(0).astype(int)
df['serious_hospitalization'] = pd.to_numeric(df['serious_hospitalization'], errors='coerce').fillna(0).astype(int)
df['serious_disabling'] = pd.to_numeric(df['serious_disabling'], errors='coerce').fillna(0).astype(int)
df['serious_life_threatening'] = pd.to_numeric(df['serious_life_threatening'], errors='coerce').fillna(0).astype(int)
print("   String to numeric conversion complete!")

# **Step 3: Standardize ages to years**
# This is tricky - the FDA data has ages in different units (years, months, weeks, days, even hours for babies).
# I need to convert everything to years so I can compare ages properly.
print("\n3. Converting patient age to years...")

def convert_age_to_years(row):
    """
    The FDA uses numeric codes for age units:
    800 = decades, 801 = years, 802 = months, 803 = weeks, 804 = days, 805 = hours
    I convert everything to years for consistency.
    """
    age = row['patient_age']
    unit = row['patient_age_unit']

    if pd.isna(age) or pd.isna(unit):
        return None

    age = float(age)

    # Convert based on the unit code
    if unit == 800:  # Decade
        return age * 10
    elif unit == 801:  # Year
        return age
    elif unit == 802:  # Month
        return age / 12
    elif unit == 803:  # Week
        return age / 52
    elif unit == 804:  # Day
        return age / 365
    elif unit == 805:  # Hour
        return age / (365 * 24)
    else:
        return None

df['patient_age_years'] = df.apply(convert_age_to_years, axis=1)
age_stats = df['patient_age_years'].describe()
print(f"   Age conversion complete. Ages range from {df['patient_age_years'].min():.1f} to {df['patient_age_years'].max():.1f} years")
print(f"   Mean age: {age_stats['mean']:.1f} years, Median age: {age_stats['50%']:.1f} years")

# **Step 4: Convert sex codes to readable labels**
# The data uses 0, 1, 2 instead of Unknown, Male, Female
print("\n4. Mapping patient sex codes...")
sex_mapping = {0: 'Unknown', 1: 'Male', 2: 'Female'}
df['patient_sex_label'] = df['patient_sex'].map(sex_mapping)
print(f"   Sex mapping complete. Value counts:")
print(df['patient_sex_label'].value_counts())

# **Step 5: Convert reporter codes to readable labels**
# Same idea - codes for who reported the adverse event need to be human-readable
print("\n5. Mapping reporter qualification codes...")
reporter_mapping = {
    1: 'Physician',
    2: 'Pharmacist',
    3: 'Other Health Professional',
    4: 'Lawyer',
    5: 'Consumer'
}
df['reporter_type'] = df['reporter_qualification'].map(reporter_mapping)
print(f"   Reporter mapping complete. Value counts:")
print(df['reporter_type'].value_counts())

# **Step 6: Create a simple "serious outcome" indicator**
# There are multiple types of serious outcomes (death, hospitalization, etc.)
# I'm creating one column that's True if ANY serious outcome happened
print("\n6. Creating serious outcome indicator...")
df['has_serious_outcome'] = (
    (df['serious_death'] == 1) |
    (df['serious_hospitalization'] == 1) |
    (df['serious_disabling'] == 1) |
    (df['serious_life_threatening'] == 1)
).astype(int)
print(f"   {df['has_serious_outcome'].sum()} reports have serious outcomes")

# **Step 7: Extract year and month for time series analysis**
# I'll want to see trends over time, so pulling out year and month makes that easier
print("\n7. Extracting year and month from dates...")
df['year'] = df['receive_date'].dt.year
df['month'] = df['receive_date'].dt.month
print(f"   Date extraction complete. Years range from {df['year'].min()} to {df['year'].max()}")


----------------------------------------------------------------------
CLEANING STEPS:
----------------------------------------------------------------------

1. Converting receive_date to datetime format...
   Conversion complete. Sample dates: [Timestamp('2014-05-22 00:00:00'), Timestamp('2014-08-21 00:00:00'), Timestamp('2014-10-14 00:00:00'), Timestamp('2015-02-11 00:00:00'), Timestamp('2015-05-11 00:00:00')]

2. Converting string columns to numeric...
   String to numeric conversion complete!

3. Converting patient age to years...
   Age conversion complete. Ages range from 0.0 to 96.0 years
   Mean age: 61.4 years, Median age: 62.0 years

4. Mapping patient sex codes...
   Sex mapping complete. Value counts:
patient_sex_label
Female     5313
Male       4334
Unknown     168
Name: count, dtype: int64

5. Mapping reporter qualification codes...
   Reporter mapping complete. Value counts:
reporter_type
Consumer                     5671
Physician                    2039
Other Health 

# 6. Checking Data Quality
Before I save this data and move on to analysis, I want to make sure it looks good.
I'll check for missing values, look at the date ranges, and verify I have data for both drugs.


In [13]:
print("\n" + "="*70)
print("DATA QUALITY SUMMARY")
print("="*70)

print("\nDataset Overview:")
print(f"Total records: {len(df)}")
print(f"Ozempic records: {len(df[df['drug'] == 'Ozempic'])}")
print(f"Metformin records: {len(df[df['drug'] == 'Metformin'])}")

print("\nMissing Values:")
missing_summary = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing_summary,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))

print("\nKey Statistics:")
print(f"Date range: {df['receive_date'].min()} to {df['receive_date'].max()}")

# Handle age stats - might be all NaN
if df['patient_age_years'].notna().any():
    print(f"Age range: {df['patient_age_years'].min():.1f} to {df['patient_age_years'].max():.1f} years")
else:
    print(f"Age range: No valid age data")

# Convert serious to numeric if needed, then count
df['serious'] = pd.to_numeric(df['serious'], errors='coerce')
serious_count = df[df['serious'] == 1].shape[0]
print(f"Serious reports: {serious_count} ({serious_count/len(df)*100:.1f}%)")
print(f"Reports with serious outcomes: {df['has_serious_outcome'].sum()} ({df['has_serious_outcome'].sum()/len(df)*100:.1f}%)")


DATA QUALITY SUMMARY

Dataset Overview:
Total records: 10000
Ozempic records: 5000
Metformin records: 5000

Missing Values:
                        Missing Count  Missing %
patient_weight                   6923      69.23
reaction_3                       5298      52.98
patient_age                      3291      32.91
patient_age_years                3291      32.91
patient_age_unit                 3291      32.91
reaction_2                       3219      32.19
country                           359       3.59
patient_sex                       185       1.85
patient_sex_label                 185       1.85
reporter_qualification            168       1.68
reporter_type                     168       1.68

Key Statistics:
Date range: 2013-10-25 00:00:00 to 2020-06-26 00:00:00
Age range: 0.0 to 96.0 years
Serious reports: 5092 (50.9%)
Reports with serious outcomes: 2936 (29.4%)


# 7. Saving the Clean Data

 Finally, I'll save this cleaned dataset as a CSV file. This way, I don't have to re-run
 the API calls every time I want to work on the project. I can just load this CSV in my
 next notebook and start analyzing immediately.


In [14]:
print("SAVING CLEANED DATA")
print("="*70)

# Save to CSV
filename = 'openfda_adverse_events_cleaned.csv'
df.to_csv(filename, index=False)
print(f"✓ Data saved to '{filename}'")
print(f"  File contains {len(df)} rows and {len(df.columns)} columns")

print("\nColumn Names and Types:")
print(df.dtypes)

print("\n" + "="*70)
print("DATA COLLECTION AND CLEANING COMPLETE!")
print("="*70)
print(f"\nCompleted at: {datetime.now()}")
print("\nWhat I accomplished:")
print("- Collected 10,000 adverse event reports from OpenFDA (5,000 each for Ozempic and Metformin)")
print("- Parsed nested JSON data into a clean tabular format")
print("- Converted dates, ages, and categorical codes into usable formats")
print("- Created additional helpful columns for analysis")
print("- Saved everything to a CSV file")


SAVING CLEANED DATA
✓ Data saved to 'openfda_adverse_events_cleaned.csv'
  File contains 10000 rows and 23 columns

Column Names and Types:
drug                                object
report_id                           object
receive_date                datetime64[ns]
serious                              int64
serious_death                        int64
serious_hospitalization              int64
serious_disabling                    int64
serious_life_threatening             int64
patient_age                        float64
patient_age_unit                   float64
patient_sex                        float64
patient_weight                      object
reaction_1                          object
reaction_2                          object
reaction_3                          object
reporter_qualification             float64
country                             object
patient_age_years                  float64
patient_sex_label                   object
reporter_type                       object
